# Bài 10 · Mạng nơ-ron và ảnh thời trang · Thực hành trên lớp (v3)

Mở notebook từ thư mục bài. Dữ liệu: `data/fashion_subset.npz` (Fashion-MNIST, Zalando Research, giấy phép MIT), cùng tập chia Bài 9: 12.000 ảnh train, 3.000 ảnh validation. **Notebook không đọc ảnh test**: test chỉ được mở một lần, sau khi chốt quyết định (slide 62).
Mã cho sẵn; việc của bạn là chạy từng ô, đối chiếu với slide và trả lời câu hỏi in đậm. Ô có dấu ✅ là ô tự kiểm.

In [1]:
from pathlib import Path
import warnings
import numpy as np
from sklearn.neural_network import MLPClassifier
ROOT = Path.cwd()
with np.load(ROOT / 'data' / 'fashion_subset.npz') as d:
    X = d['train_images'].astype(np.float64) / 255; y = d['train_labels'].astype(int)
    V = d['validation_images'].astype(np.float64) / 255; yv = d['validation_labels'].astype(int)
TEN = ['áo phông', 'quần dài', 'áo len', 'váy', 'áo khoác', 'dép xăng-đan', 'áo sơ mi', 'giày thể thao', 'túi', 'bốt cổ ngắn']
XOR = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], float); YX = np.array([0, 1, 1, 0])
sigmoid = lambda s: 1 / (1 + np.exp(-s))
bce = lambda s, t: float(np.mean(np.logaddexp(0, s) - t * s))
print(X.shape, V.shape)

(12000, 784) (3000, 784)


## Chặng 1 · Một lớp tuyến tính: làm được gì, hỏng ở đâu (slide 5–17)

Thử 200.000 đường thẳng ngẫu nhiên s = w₁x₁ + w₂x₂ + b trên bốn điểm XOR: đếm số điểm đúng tốt nhất, và kiểm đẳng thức s(0;0) + s(1;1) = s(0;1) + s(1;0).

In [2]:
rng = np.random.default_rng(0)
w = rng.normal(size=(200_000, 3))                      # cột: w1, w2, b
S = np.c_[XOR, np.ones(4)] @ w.T                       # 4 × 200.000 score
dung = ((S >= 0) == YX[:, None].astype(bool)).sum(0)
print('tốt nhất:', dung.max(), '/4 · tỷ lệ đường đúng 3/4:', round(np.mean(dung == 3), 3))

tốt nhất: 3 /4 · tỷ lệ đường đúng 3/4: 0.127


In [3]:
# ✅ TỰ KIỂM · chặng 1
assert dung.max() == 3
assert np.allclose(S[0] + S[3], S[1] + S[2])
print('✅ không đường thẳng nào đúng cả bốn; đẳng thức affine đúng với mọi (w1, w2, b) đã thử (slide 12–13)')

✅ không đường thẳng nào đúng cả bốn; đẳng thức affine đúng với mọi (w1, w2, b) đã thử (slide 12–13)


**Câu hỏi:** thêm cột đặc trưng x₁·x₂ vào `np.c_[XOR, np.ones(4)]` thì tốt nhất là bao nhiêu? Thử lại.

## Chặng 2 · Tầng ẩn: đổi biểu diễn (slide 18–33)

Tính tay tầng ẩn h = ReLU(Wx + b) với W = [[1, 1], [1, 1]], b = (0; −1), rồi tầng ra v = (2; −4), c = −1. Sau đó bỏ ReLU để so sánh.

In [4]:
W = np.array([[1, 1], [1, 1]]); b = np.array([0, -1]); v = np.array([2, -4]); c = -1
A = XOR @ W.T + b
H = np.maximum(A, 0)
s = H @ v + c
s_khong_relu = A @ v + c
print('H =', H.tolist(), '· s =', s.tolist(), '· bỏ ReLU:', s_khong_relu.tolist(), '· loss =', round(bce(s, YX), 4))

H = [[0.0, 0.0], [1.0, 0.0], [1.0, 0.0], [2.0, 1.0]] · s = [-1.0, 1.0, 1.0, -1.0] · bỏ ReLU: [3.0, 1.0, 1.0, -1.0] · loss = 0.3133


In [5]:
# ✅ TỰ KIỂM · chặng 2
assert H.tolist() == [[0, 0], [1, 0], [1, 0], [2, 1]] and s.tolist() == [-1, 1, 1, -1] and s_khong_relu.tolist() == [3, 1, 1, -1]
assert abs(bce(s, YX) - 0.3132616875) < 1e-9
so_tham_so = lambda tang: sum(a * b_ + b_ for a, b_ in zip(tang[:-1], tang[1:]))
assert so_tham_so([2, 2, 1]) == 9 and so_tham_so([784, 64, 10]) == 50890 and so_tham_so([784, 32, 10]) == 25450
print('✅ bảng tầng ẩn, score, loss và số tham số đúng slide 20, 23, 25, 29')

✅ bảng tầng ẩn, score, loss và số tham số đúng slide 20, 23, 25, 29


In [6]:
# Cộng ReLU dựng đường cong f(x) = sin 2x + 0,3x trên [0; 3] (slide 27)
f = lambda q: np.sin(2 * q) + 0.3 * q
xs = np.linspace(0, 3, 2001)
def xap_xi(K):
    t = 3 * np.arange(K + 1) / K; fv = f(t); doc = np.diff(fv) / np.diff(t)
    he_so = np.r_[doc[0], np.diff(doc)]
    return fv[0] + sum(he_so[k] * np.maximum(xs - t[k], 0) for k in range(K))
lech = {K: float(np.max(np.abs(xap_xi(K) - f(xs)))) for K in (2, 4, 8, 16)}
print({K: round(e, 4) for K, e in lech.items()})

{2: 0.9272, 4: 0.2317, 8: 0.0686, 16: 0.0175}


In [7]:
# ✅ TỰ KIỂM · cộng ReLU
assert abs(lech[16] - 0.0175203523) < 1e-9 and lech[2] > lech[4] > lech[8] > lech[16]
print('✅ lệch tối đa giảm khi thêm ReLU, khớp slide 27')

✅ lệch tối đa giảm khi thêm ReLU, khớp slide 27


**Câu hỏi:** mạng 1 → 16 → 1 cần bao nhiêu tham số để biểu diễn đường gãy 16 khúc này?

## Chặng 3 · Học: loss, gradient, lan truyền ngược (slide 35–51)

Ví dụ tay ở slide 42–45: x = (0,6; 0,2), nhãn 1, chín tham số cho sẵn. Viết forward và backward, rồi kiểm bằng sai phân.

In [8]:
theta = np.array([.5, -.4, .3, .8, .1, .1, .7, -.5, .2]); x = np.array([.6, .2]); nhan = 1
def forward(t):
    Wt = t[:4].reshape(2, 2); bt = t[4:6]; vt = t[6:8]; ct = t[8]
    a = Wt @ x + bt; h = np.maximum(a, 0); st = vt @ h + ct
    return a, h, st, bce(np.array([st]), np.array([nhan]))
def backward(t):
    a, h, st, _ = forward(t)
    delta = sigmoid(st) - nhan                    # ∂L/∂s = p − y
    dh = delta * t[6:8] * (a > 0)                 # qua ReLU
    return np.r_[np.outer(dh, x).ravel(), dh, delta * h, delta]
g = backward(theta)
so = np.array([(forward(theta + e)[3] - forward(theta - e)[3]) / 2e-6 for e in np.eye(9) * 1e-6])
print('gradient:', g.round(4)); print('chênh lớn nhất so với sai phân:', np.abs(g - so).max())

gradient: [-0.1887 -0.0629  0.1348  0.0449 -0.3144  0.2246 -0.1437 -0.1976 -0.4492]
chênh lớn nhất so với sai phân: 7.549356972891275e-11


In [9]:
# ✅ TỰ KIỂM · lan truyền ngược
assert abs(forward(theta)[2] - 0.204) < 1e-12 and abs(g[8] - (-0.449176135033)) < 1e-10
assert np.abs(g - so).max() < 1e-8
print('✅ s = 0,204; δ = p − y; lan truyền ngược khớp sai phân (slide 42–45)')

✅ s = 0,204; δ = p − y; lan truyền ngược khớp sai phân (slide 42–45)


In [10]:
# Huấn luyện XOR từ khởi tạo ngẫu nhiên (slide 47–48): cùng khởi tạo và cùng thuật toán với mã của bài
def chay_xor(seed, H=2, buoc=2000, eta=0.5):
    r = np.random.default_rng(seed)
    W1 = r.normal(0, np.sqrt(2 / 2), (2, H)); b1 = np.zeros(H); W2 = r.normal(0, np.sqrt(1 / H), H); b2 = 0.0
    for _ in range(buoc):
        A_ = XOR @ W1 + b1; H_ = np.maximum(A_, 0); s_ = H_ @ W2 + b2
        d_ = (sigmoid(s_) - YX) / 4; dA = d_[:, None] * W2 * (A_ > 0)
        W1 -= eta * XOR.T @ dA; b1 -= eta * dA.sum(0); W2 -= eta * H_.T @ d_; b2 -= eta * d_.sum()
    s_ = np.maximum(XOR @ W1 + b1, 0) @ W2 + b2
    return bce(s_, YX), int(((s_ >= 0) == YX.astype(bool)).sum())
print('seed 1:', chay_xor(1), '· seed 0:', chay_xor(0))
ty_le = {H: sum(chay_xor(s, H)[1] == 4 for s in range(20)) for H in (2, 8)}
print('đúng 4/4 trong 20 seed đầu:', ty_le)

seed 1: (0.001559249649033424, 4) · seed 0: (0.34682099224031454, 3)


đúng 4/4 trong 20 seed đầu: {2: 2, 8: 20}


In [11]:
# ✅ TỰ KIỂM · huấn luyện XOR
assert abs(chay_xor(1)[0] - 0.001559249649) < 1e-9 and chay_xor(0)[1] < 4
assert ty_le[8] > ty_le[2]
print('✅ seed 1 thành công, seed 0 kẹt; thêm đơn vị ẩn làm tỷ lệ thành công tăng (slide 47–48)')

✅ seed 1 thành công, seed 0 kẹt; thêm đơn vị ẩn làm tỷ lệ thành công tăng (slide 47–48)


**Câu hỏi:** in tiền kích hoạt A_ của seed 0 sau khi huấn luyện. Hai điểm nào có cả hai ReLU đều tắt?

## Chặng 4 · Mạng trên ảnh thật (slide 52–69)

Huấn luyện MLP 784 → 64 → 10 đúng cấu hình của giao thức nhưng chỉ 10 epoch (khoảng 2 giây), rồi nạp trọng số checkpoint (epoch 40) đã lưu để tính tay forward.

In [12]:
mlp = MLPClassifier(hidden_layer_sizes=(64,), activation='relu', solver='adam', alpha=1e-4, batch_size=256, learning_rate_init=1e-3, random_state=42)
for ep in range(10):
    with warnings.catch_warnings():
        warnings.simplefilter('ignore'); mlp.partial_fit(X, y, classes=np.arange(10))
dung10 = int((mlp.predict(V) == yv).sum())
print('sau 10 epoch: đúng', dung10, '/ 3.000 ảnh validation')

sau 10 epoch: đúng 2546 / 3.000 ảnh validation


In [13]:
# ✅ TỰ KIỂM · huấn luyện ngắn
assert abs(dung10 - 2546) <= 3          # cùng quy trình với mã của bài (epoch 10)
assert dung10 > 2501                # đã vượt logistic (2501)
print('✅ sau 10 epoch đã vượt logistic; khớp đường ở slide 54')

✅ sau 10 epoch đã vượt logistic; khớp đường ở slide 54


In [14]:
with np.load(ROOT / 'data' / 'mlp_arrays_v3.npz') as A_:
    W1, b1, W2, b2 = A_['W1'], A_['b1'], A_['W2'], A_['b2']
def softmax(Z):
    Z = Z - Z.max(1, keepdims=True); E = np.exp(Z); return E / E.sum(1, keepdims=True)
Hv = np.maximum(V @ W1 + b1, 0)                      # 3.000 × 64
Pv = softmax(Hv @ W2 + b2)
dung = int((Pv.argmax(1) == yv).sum())
i = 60                                               # ảnh áo sơ mi ở slide 1
print('đúng', dung, '/ 3.000 · ảnh', i, ':', (Hv[i] > 0).sum(), 'đơn vị bật · xác suất áo sơ mi', round(Pv[i, 6], 3))
def dich(M, dx):
    S_ = np.zeros_like(M).reshape(-1, 28, 28); S_[:, :, dx:] = M.reshape(-1, 28, 28)[:, :, :-dx]; return S_.reshape(-1, 784)
dung_dich2 = int((softmax(np.maximum(dich(V, 2) @ W1 + b1, 0) @ W2 + b2).argmax(1) == yv).sum())
print('dịch 2 pixel: đúng', dung_dich2)

đúng 2581 / 3.000 · ảnh 60 : 39 đơn vị bật · xác suất áo sơ mi 0.518
dịch 2 pixel: đúng 1640


In [15]:
# ✅ TỰ KIỂM · checkpoint và dịch ảnh
assert dung == 2581 and (Hv[60] > 0).sum() == 39
assert abs(Pv[60, 6] - 0.5175780580) < 1e-9
assert abs(dung_dich2 / 3000 - 0.5466666667) < 1e-9
print('✅ forward tự viết trùng checkpoint của bài (slide 54, 56); dịch 2 pixel làm accuracy sụp (slide 66)')

✅ forward tự viết trùng checkpoint của bài (slide 54, 56); dịch 2 pixel làm accuracy sụp (slide 66)


**Câu hỏi cuối:** vì sao notebook này không đọc ảnh test? Nếu bạn thử 10 cấu hình rồi chọn cấu hình tốt nhất trên test, con số báo cáo sai ở đâu?